In [13]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage


# --------------------------------------------------
# LLM: LangChain OpenAI (NOT ChatOpenAI)
# --------------------------------------------------
llm = ChatOpenAI(
    model="qwen/qwen3-vl-235b-a22b-thinking",
    openai_api_key="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",
    openai_api_base="https://openrouter.ai/api/v1",
    extra_body={
        "reasoning": {"enabled": True}
    },
    temperature=0
)


# --------------------------------------------------
# Prompt Template
# --------------------------------------------------
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a careful assistant. Reason step by step before answering."),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}")
    ]
)


# --------------------------------------------------
# Conversation Memory
# --------------------------------------------------
chat_history = []


# --------------------------------------------------
# Serialize messages → text prompt
# --------------------------------------------------
def messages_to_text(messages):
    text = []
    for msg in messages:
        if isinstance(msg, HumanMessage):
            text.append(f"User: {msg.content}")
        elif isinstance(msg, AIMessage):
            text.append(f"Assistant: {msg.content}")
    return "\n".join(text)


# --------------------------------------------------
# Chat Function
# --------------------------------------------------
def chat(user_input: str) -> str:
    global chat_history

    messages = prompt.format_messages(
        chat_history=chat_history,
        input=user_input
    )

    #prompt_text = messages_to_text(messages)

    response = llm.invoke(messages)

    # IMPORTANT: cast to string
    response_text = str(response.content)

    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=str(response)))

    return response_text
# --------------------------------------------------
# Run Chat App
# --------------------------------------------------
if __name__ == "__main__":
    print("🔹 LangChain OpenAI (non-chat) + OpenRouter")
    print("Type 'exit' to quit\n")

    while True:
        user_input = input("User: ")

        if user_input.lower() in {"exit", "quit"}:
            print("Goodbye 👋")
            break

        reply = chat(user_input)
        print("Assistant:", reply)
        print()

🔹 LangChain OpenAI (non-chat) + OpenRouter
Type 'exit' to quit

User: Hi
Assistant: Hi! How can I assist you today? 😊

User: What is my name
Assistant: I don't have access to your name unless you've shared it with me during our conversation. Since you haven't provided your name yet, I can't know it. If you'd like me to use your name in our conversation, feel free to share it with me! 😊



KeyboardInterrupt: Interrupted by user

## LCEL and OpenAI and Prompt Template with History

In [4]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnableLambda

llm = ChatOpenAI(
    model="qwen/qwen3-vl-235b-a22b-thinking",
    openai_api_key="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",
    openai_api_base="https://openrouter.ai/api/v1",
    extra_body={
        "reasoning": {"enabled": True}
    },
    temperature=0
)

analysis_prompt = ChatPromptTemplate.from_messages(
    [
      ("system", "You are a smart analysis agent. Study the history and the user prompt and come with a step by step analysis"),
      MessagesPlaceholder(variable_name="chat_history"),
      ("human", "{user_input}")
    ]
  )

final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system","You have the analysis and chat history. Now come up with the answer."),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "Analysis {analysis}")
    ]
)

chat_history = []

def extract_analysis(msg):
    return {"analysis": msg.content,"chat_history":chat_history}

analysis_step = RunnableLambda(extract_analysis)

chain = analysis_prompt | llm | analysis_step | final_prompt | llm

def chat(user_input: str) -> str:
    global chat_history

    answer = chain.invoke({
        "chat_history": chat_history,
        "user_input": user_input
    })

    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=str(answer)))

    return answer.content


if __name__ == "__main__":
    print("🔹 LangChain OpenAI (non-chat) + OpenRouter")
    print("Type 'exit' to quit\n")

    while True:
        user_input = input("User: ")

        if user_input.lower() in {"exit", "quit"}:
            print("Goodbye 👋")
            break

        reply = chat(user_input)
        print("Assistant:", reply)
        print()




🔹 LangChain OpenAI (non-chat) + OpenRouter
Type 'exit' to quit

User: Hi, my name is Jibitesh
Assistant: Hi Jibitesh! It's great to meet you. How can I assist you today? I'm here to help with anything you need—just let me know! 😊

User: What is 2+2 ?
Assistant: That's absolutely correct! 😊 2 + 2 is indeed **4**. Great job! Let me know if you'd like to explore more math problems or need help with anything else—I'm here to assist!

User: Where is Washington DC ?
Assistant: Thanks for sharing this detailed analysis of Washington, D.C.! You've covered the key points perfectly—it’s a federal district established in 1790, not part of any state, and its unique status as the nation’s capital is well explained. The geographic context (Potomac River, Maryland/Virginia borders) and political nuances (non-voting delegate in Congress) are spot-on. 🌟  

Let me know if you’d like to explore related topics, like the history of D.C.’s founding, its cultural significance, or even fun facts about landmar

## RAG

In [1]:
!pip install -U langchain-text-splitters

In [3]:
!pip install -U langchain-community

In [2]:
!pip install -U \
  langchain-core \
  langchain-openai \
  langchain-community \
  langchain-text-splitters \
  langchain-huggingface \
  sentence-transformers \
  faiss-cpu

INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.2/502.2 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 98.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installatio

In [5]:
import os
from typing import List

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnableLambda
from langchain_core.documents import Document

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS


# =========================================================
# 1️⃣ LLM CONFIG (same style as your original app)
# =========================================================

llm = ChatOpenAI(
    model="qwen/qwen3-vl-235b-a22b-thinking",
    openai_api_key="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",
    openai_api_base="https://openrouter.ai/api/v1",
    extra_body={"reasoning": {"enabled": True}},
    temperature=0
)


# =========================================================
# 2️⃣ KNOWLEDGE BASE (replace with files later)
# =========================================================

raw_docs: List[Document] = [
    Document(page_content="""
LangChain Expression Language (LCEL) allows developers to compose
AI pipelines using the pipe (|) operator. It is deterministic and debuggable.
"""),
    Document(page_content="""
Retrieval-Augmented Generation (RAG) improves LLM accuracy by retrieving
relevant chunks of information from external knowledge sources.
"""),
    Document(page_content="""
FAISS is a vector similarity search library used to build fast
in-memory vector databases for retrieval.
""")
]


# =========================================================
# 3️⃣ CHUNKING (MANDATORY FOR RAG)
# =========================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks: List[Document] = text_splitter.split_documents(raw_docs)


# =========================================================
# 4️⃣ EMBEDDINGS (NON-OPENAI, NON-DEPRECATED)
# =========================================================

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


# =========================================================
# 5️⃣ VECTOR STORE + RETRIEVER
# =========================================================

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})


# =========================================================
# 6️⃣ RETRIEVAL RUNNABLE (LCEL STYLE)
# =========================================================

def retrieve_context(inputs):
    query = inputs["user_input"]
    chat_history = inputs["chat_history"]

    retrieved_docs = retriever.invoke(query)

    context = "\n\n".join(
        f"[Chunk {i}]\n{doc.page_content}"
        for i, doc in enumerate(retrieved_docs)
    )

    return {
        "context": context,
        "user_input": query,
        "chat_history": chat_history
    }

retrieval_step = RunnableLambda(retrieve_context)


# =========================================================
# 7️⃣ ANALYSIS PROMPT
# =========================================================

analysis_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a smart assistant.\n"
            "Use the retrieved context when it is relevant.\n"
            "If the question is conversational or based on chat history "
            "(e.g., user's name, greetings, clarifications), answer normally.\n"
            "Do NOT hallucinate external factual information.\n\n"
            "Context (if any):\n{context}"
        ),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{user_input}")
    ]
)

# =========================================================
# 8️⃣ ANALYSIS EXTRACTION
# =========================================================
chat_history: List = []
def extract_analysis(msg):
    return {"analysis": msg.content, "chat_history":chat_history}

analysis_step = RunnableLambda(extract_analysis)


# =========================================================
# 9️⃣ FINAL PROMPT
# =========================================================

final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You have the analysis. Produce a clear final answer."),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{analysis}")
    ]
)


# =========================================================
# 🔗 10️⃣ FULL LCEL RAG CHAIN
# =========================================================

chain = (
    retrieval_step
    | analysis_prompt
    | llm
    | analysis_step
    | final_prompt
    | llm
)


# =========================================================
# 11️⃣ CHAT LOOP (HISTORY PRESERVED)
# =========================================================



def chat(user_input: str) -> str:
    global chat_history

    answer = chain.invoke({
        "user_input": user_input,
        "chat_history": chat_history
    })

    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=answer.content))

    return answer.content


# =========================================================
# 12️⃣ CLI ENTRYPOINT
# =========================================================

if __name__ == "__main__":
    print("🔹 LCEL RAG Application (Stable & Correct)")
    print("Type 'exit' to quit\n")

    while True:
        user_input = input("User: ")

        if user_input.lower() in {"exit", "quit"}:
            print("Goodbye 👋")
            break

        reply = chat(user_input)
        print("\nAssistant:", reply)
        print()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

🔹 LCEL RAG Application (Stable & Correct)
Type 'exit' to quit

User: Hi, my name is Jibitesh

Assistant: Hi there! 😊 I think there might be a little mix-up—I'm Qwen, the large language model developed by Tongyi Lab, not Jibitesh. But I'm happy to help! How can I assist you today? Feel free to ask me anything! 🌟

User: What is RAG ?

Assistant: That's an **excellent and precise explanation** of RAG! 🙌 You’ve perfectly captured its core purpose:  
> *"Retrieval-Augmented Generation (RAG) improves LLM accuracy by retrieving relevant chunks of information from external knowledge sources."*  

### Key Takeaways from Your Description:
1. **Solves LLM Limitations**:  
   RAG addresses issues like *factual hallucinations* or *outdated knowledge* by pulling real-time/data-specific info from external sources (e.g., databases, documents).

2. **Hybrid Approach**:  
   - **Retrieval**: Finds relevant context (e.g., via vector search).  
   - **Generation**: Uses the retrieved context to craft accu